In [1]:
"""
data_utils.py
────────────────────────────────────────────────────────────────────────────────
数据工具层（Data Utilities）

职责：
  1. 懒加载三张表并全部常驻内存。
  2. 寻找时空锚点：给定 T_target，用二分查找 O(logN) 定位 level2 中
     <= T_target 的最后一行快照。
  3. 提取与缝合流水：给定 (T_start, T_end]，切出 order + trade，
     打标签、合并，将时间戳提为列后统一 sort_values 排序，再还原为索引。

排序规则：
  - 主键：时间戳升序
  - 同一时间戳：有 seq_no 的按 seq_no 升序；无 seq_no 的 order 优先于 trade
────────────────────────────────────────────────────────────────────────────────
"""

import os
import pandas as pd

# 同时间戳时，event_type 的排序权重（数值越小越靠前）
_EVENT_PRIORITY = {"order": 0, "trade": 1}


class DataLoader:
    """
    负责从 clean_data 目录读取 Parquet 文件，并提供查询接口。

    Parameters
    ----------
    clean_dir : str
        clean_data 文件夹的路径，包含：
        level2.parquet / order.parquet / trade.parquet
    """

    def __init__(self, clean_dir: str):
        self.clean_dir = clean_dir

        # 三张表均懒加载常驻内存
        self._df_level2: pd.DataFrame | None = None
        self._df_order:  pd.DataFrame | None = None
        self._df_trade:  pd.DataFrame | None = None

    # ── 1. 懒加载（三张表统一模式）───────────────────────────────────────────
    def _lazy_load(self, attr: str, filename: str) -> pd.DataFrame:
        """通用懒加载：首次访问时读取 Parquet 并常驻内存。"""
        if getattr(self, attr) is None:
            path = os.path.join(self.clean_dir, filename)
            df   = pd.read_parquet(path, engine="pyarrow")
            setattr(self, attr, df)
            print(f"[DataLoader] {filename} 已加载：{len(df):,} 行")
        return getattr(self, attr)

    @property
    def level2(self) -> pd.DataFrame:
        return self._lazy_load("_df_level2", "level2.parquet")

    @property
    def order(self) -> pd.DataFrame:
        return self._lazy_load("_df_order", "order.parquet")

    @property
    def trade(self) -> pd.DataFrame:
        return self._lazy_load("_df_trade", "trade.parquet")

    # ── 2. 寻找时空锚点（二分查找 O(logN)）──────────────────────────────────
    def find_anchor(self, t_target: int) -> pd.Series:
        """
        在 level2 中用二分查找定位时间戳 <= t_target 的最后一行快照。

        Parameters
        ----------
        t_target : int
            目标时间戳，格式同 level2 索引（如 93003000）。

        Returns
        -------
        pd.Series
            满足条件的最后一行 level2 快照。

        Raises
        ------
        ValueError
            若 t_target 早于 level2 中的最早快照，无法找到锚点。
        """
        df  = self.level2
        # searchsorted side='right' 找到第一个 > t_target 的位置，减 1 即为 <= t_target 的最后一个
        idx = df.index.searchsorted(t_target, side="right") - 1

        if idx < 0:
            raise ValueError(
                f"[DataLoader] 找不到锚点：t_target={t_target} 早于 level2 最早时间 {df.index[0]}"
            )

        return df.iloc[idx]

    # ── 3. 提取与缝合流水 ─────────────────────────────────────────────────────
    def get_events(self, t_start: int, t_end: int) -> pd.DataFrame:
        """
        从 order 和 trade 表中切出 (T_start, T_end] 区间的数据，
        打上 event_type 标签后合并，并按时间戳 + 优先级严格排序。

        Parameters
        ----------
        t_start : int
            区间左端（开区间，不含），格式同 level2 索引。
        t_end : int
            区间右端（闭区间，含），格式同 level2 索引。

        Returns
        -------
        pd.DataFrame
            合并后的事件流，列包含原始列与 event_type，
            索引为时间戳（int），严格升序。
        """
        # ── 切片 order ────────────────────────────────────────────────────────
        df_o = self.order
        df_order = df_o.loc[
            (df_o.index > t_start) & (df_o.index <= t_end)
        ].copy()
        df_order["event_type"] = "order"

        # ── 切片 trade ────────────────────────────────────────────────────────
        df_t = self.trade
        df_trade = df_t.loc[
            (df_t.index > t_start) & (df_t.index <= t_end)
        ].copy()
        df_trade["event_type"] = "trade"

        # ── 纵向拼接 ──────────────────────────────────────────────────────────
        combined = pd.concat([df_order, df_trade], axis=0, sort=False)

        if combined.empty:
            return combined

        # ── 排序：将时间戳提为列，与优先级在同一个 sort_values 里统一排序 ────
        combined["_time"]     = combined.index
        combined["_priority"] = combined["event_type"].map(_EVENT_PRIORITY)

        if "seq_no" in combined.columns:
            # seq_no 存在时最优先；NaN 排在有序列号之后
            combined["_seq"] = combined["seq_no"].fillna(float("inf"))
            sort_keys = ["_time", "_seq", "_priority"]
        else:
            sort_keys = ["_time", "_priority"]

        combined.sort_values(sort_keys, kind="stable", inplace=True)

        # 还原时间戳为索引，清理临时列
        combined.set_index("_time", inplace=True)
        combined.index.name = "time"

        drop_cols = ["_priority"]
        if "_seq" in combined.columns:
            drop_cols.append("_seq")
        combined.drop(columns=drop_cols, inplace=True)

        return combined